# SNNs auf GPUs -- Custom CUDA Kernel (Colab)

**Before running:** Runtime -> Change runtime type -> T4 GPU -> Save

Then run cells top to bottom.

In [ ]:
# Cell 1 -- Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Runtime -> Change runtime type -> T4 GPU")

props = torch.cuda.get_device_properties(0)
print("GPU        :", props.name)
print("VRAM       :", round(props.total_memory / 1e9, 1), "GB")
print("SM arch    : sm_" + str(props.major) + str(props.minor))
print("CUDA ver   :", torch.version.cuda)


In [ ]:
# Cell 2 -- Clone repo and switch to Operators branch
import os

REPO = "/content/SNNs-auf-GPUs"
if not os.path.exists(REPO):
    os.system("git clone https://github.com/Zuzu3290/SNNs-auf-GPUs.git " + REPO)

os.chdir(REPO)
os.system("git checkout Operators")
os.system("git pull origin Operators")
print("Working directory:", os.getcwd())


In [ ]:
# Cell 3 -- Install Python dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tonic", "norse", "spikingjelly", "pyyaml"], check=True)
print("Dependencies installed.")


In [ ]:
# Cell 4 -- JIT-compile the CUDA extension
# Uses torch.utils.cpp_extension.load (content-hash cache) instead of
# setup.py build_ext (timestamp cache that silently reuses stale .so files)
import os, sys
from torch.utils.cpp_extension import load

os.chdir("/content/SNNs-auf-GPUs")
ROOT   = "/content/SNNs-auf-GPUs"
KERN   = ROOT + "/src/crsc/kernels"
ATTRS  = ROOT + "/acceleration/GPU_attributes"

lif = load(
    name            = "snn_forward",
    sources         = [
        KERN  + "/snn_binding.cpp",
        KERN  + "/snn_forward.cu",
        KERN  + "/lif_temporal.cu",
        KERN  + "/lif_warp_oriented.cu",
        ATTRS + "/energy_management.cu",
        ATTRS + "/memory_management.cu",
        ATTRS + "/throughput_optimiation.cu",
    ],
    extra_cflags      = ["-O3", "-DSNN_HAS_NVML=0"],
    extra_cuda_cflags = ["-O3", "--use_fast_math", "-DSNN_HAS_NVML=0"],
    extra_include_paths = [ATTRS],
    verbose = True,
)

print("Functions:", [x for x in dir(lif) if not x.startswith("_")])


In [ ]:
# Cell 5 -- Smoke test (lif is already loaded from Cell 4)
import torch

B, N, T = 4, 512, 25
inp     = torch.rand(B, N, T, device="cuda")
voltage = torch.zeros(B, N,   device="cuda")

spikes = lif.forward(inp, voltage)
print("Standard forward  shape:", tuple(spikes.shape))

spikes = lif.temporal_forward(inp, voltage)
print("Temporal forward  shape:", tuple(spikes.shape))

spikes, blocks, npt = lif.warp_oriented_forward(inp, voltage)
print("Warp-oriented     shape:", tuple(spikes.shape),
      "  blocks:", blocks, "  npt:", round(npt, 2))


In [ ]:
# Cell 6 -- Profiled forward: kernel time + energy
voltage.zero_()

result = lif.forward_profiled(inp, voltage, v_th=1.0, tau_inv=0.1)
spikes, elapsed_ms, pwr_before, pwr_after, energy_mj, nvml_ok = result

print("Profiled forward")
print("  Kernel time : {:.3f} ms".format(elapsed_ms))
print("  Spike rate  : {:.1f}%".format(spikes.mean().item() * 100))
if nvml_ok:
    print("  Power before: {:.1f} mW".format(pwr_before))
    print("  Power after : {:.1f} mW".format(pwr_after))
    print("  Energy used : {:.4f} mJ".format(energy_mj))
else:
    print("  Power/Energy: N/A (NVML not available on this GPU tier)")


In [ ]:
# Cell 7 -- Benchmark: warp-oriented kernel vs PyTorch baseline
# warp_oriented_forward is async (no CPU-GPU sync per call) -- fair comparison
import time

RUNS  = 200
WARMUP = 20

# --- Warm up GPU --------------------------------------------------
voltage.zero_()
for _ in range(WARMUP):
    lif.warp_oriented_forward(inp, voltage)
torch.cuda.synchronize()

# --- Warp-oriented kernel (async) ---------------------------------
voltage.zero_()
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(RUNS):
    lif.warp_oriented_forward(inp, voltage)
torch.cuda.synchronize()  # one sync at the end, not per call
kernel_ms = (time.perf_counter() - t0) * 1000 / RUNS

# --- PyTorch baseline (same async pattern) ------------------------
def lif_torch(x, v, v_th=1.0, tau_inv=0.1):
    v_new = v * (1.0 - tau_inv) + x
    spk   = (v_new >= v_th).float()
    return spk, v_new * (1.0 - spk)

v_pt = torch.zeros(B, N, device="cuda")
for _ in range(WARMUP):
    for t in range(T):
        _, v_pt = lif_torch(inp[:, :, t], v_pt)
torch.cuda.synchronize()

v_pt.zero_()
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(RUNS):
    for t in range(T):
        _, v_pt = lif_torch(inp[:, :, t], v_pt)
torch.cuda.synchronize()
torch_ms = (time.perf_counter() - t0) * 1000 / RUNS

# --- Timed kernel measurement (pure kernel time, no Python overhead) --
voltage.zero_()
_, elapsed_ms, blocks, npt = lif.warp_oriented_timed(inp, voltage)

print("Warp-oriented kernel (wall) : {:.3f} ms / forward".format(kernel_ms))
print("PyTorch baseline   (wall)  : {:.3f} ms / forward".format(torch_ms))
print("Speedup                    : {:.2f}x".format(torch_ms / kernel_ms))
print("")
print("Pure kernel time (GPU only): {:.4f} ms".format(elapsed_ms))
print("Blocks launched            : {}".format(blocks))
print("Neurons per thread         : {:.2f}".format(npt))


In [ ]:
# Cell 8 -- (Optional) Full training run with kernel ON
import yaml, subprocess, sys
from pathlib import Path

cfg_path = Path("/content/SNNs-auf-GPUs/SNN_module.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["training"]["kernel"]               = "ON"
cfg["training"]["epochs"]               = 1
cfg["training"]["iterations_per_epoch"] = 20
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False))
print("SNN_module.yaml updated: kernel=ON, 1 epoch, 20 iterations")

subprocess.run(
    [sys.executable, "src/learning/main.py", "--model", "torch"],
    cwd="/content/SNNs-auf-GPUs"
)
